# Assignment 3 — OLMo-2 / PopQA experiment

This notebook runs the supplied project on a Colab GPU. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another GPU) before running. It will upload your ZIP, run the four OLMo-2 conditions, run the analysis, and package the generated predictions/results. Model inference over ~3,003 examples per condition may take a while and use substantial GPU time.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next((n for n in uploaded if n.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Upload Assignment3_code_and_results.zip')


In [ ]:
import os, zipfile, shutil, pathlib
ROOT = '/content/assignment3'
shutil.rmtree(ROOT, ignore_errors=True)
os.makedirs(ROOT, exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(ROOT)
print('Project extracted to', ROOT)
print('GPU check:')


In [ ]:
!nvidia-smi

In [ ]:
%cd /content/assignment3
!pip -q install 'transformers>=4.45,<5' accelerate pandas numpy scipy statsmodels pytest


## 1. Validate the supplied scorer and prompt

In [ ]:
%cd /content/assignment3/src
!python -m pytest -v test_scorer.py
!python run_llm.py --dry-run


## 2. Run each model condition

The runner writes `results/preds_base.jsonl`, `preds_sft.jsonl`, `preds_dpo.jsonl`, and `preds_instruct.jsonl`. If a session is interrupted, rerun a condition only if its output file is incomplete; each run overwrites that condition's file.

In [ ]:
import os, subprocess, time
conditions = ['base', 'sft', 'dpo', 'instruct']
for condition in conditions:
    output = f'/content/assignment3/results/preds_{condition}.jsonl'
    print(f'\n===== Running {condition} =====', flush=True)
    start = time.time()
    subprocess.run(['python', 'run_llm.py', '--condition', condition, '--prompt_set', '1', '--batch', '8'], check=True)
    print(f'{condition} finished in {(time.time()-start)/60:.1f} minutes; rows={sum(1 for _ in open(output))}', flush=True)


## 3. Analyze the four conditions

In [ ]:
%cd /content/assignment3/src
!python analyze.py base,sft,dpo,instruct | tee ../logs/llm_analysis.log


## 4. Package outputs for submission

This creates a ZIP with all project files plus the model predictions and analysis log. Download it and use the measured results in the report; do not treat the prior-frequency baseline as an LLM result.

In [ ]:
%cd /content
import os, zipfile
out_zip = '/content/Assignment3_with_llm_results.zip'
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for base, dirs, files_ in os.walk('/content/assignment3'):
        dirs[:] = [d for d in dirs if d not in ['.git', '__pycache__']]
        for f in files_:
            path = os.path.join(base, f)
            z.write(path, os.path.relpath(path, '/content/assignment3'))
print('Created', out_zip, 'size MB:', round(os.path.getsize(out_zip)/1e6, 2))
from google.colab import files
files.download(out_zip)
